## 1. Prepare Environment And Dataset


In [1]:
# Grant the huggingface token
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [2]:
# Set up the GPU
import torch

if torch.cuda.is_available():
    device = "cuda:0"
    # load the model in half-precision (float16) if running on a GPU
    # this will speed up inference at almost no cost to WER accuracy.
    torch_dtype = torch.float16
else:
    device = "cpu"
    torch_dtype = torch.float32
# check what process we use:
print(f"device: {device}, torch_dtype: {torch_dtype}")

device: cuda:0, torch_dtype: torch.float16


In [15]:
# Load dataset
from datasets import load_dataset, DatasetDict

audio_data = DatasetDict()

audio_data["train"] = load_dataset(
    "speechcolab/gigaspeech", "xs",
    token=True, streaming=True
)["validation"].take(100)

audio_data["test"] = load_dataset(
    "speechcolab/gigaspeech", "xs",
    token=True, streaming=True
)["test"].take(100)

print(audio_data)
# IterableDataset generated because streaming=True

DatasetDict({
    train: IterableDataset({
        features: ['segment_id', 'speaker', 'text', 'audio', 'begin_time', 'end_time', 'audio_id', 'title', 'url', 'source', 'category', 'original_full_path'],
        num_shards: 1
    })
    test: IterableDataset({
        features: ['segment_id', 'speaker', 'text', 'audio', 'begin_time', 'end_time', 'audio_id', 'title', 'url', 'source', 'category', 'original_full_path'],
        num_shards: 2
    })
})


In [16]:
# Only keep the audio and target text

audio_data["train"] = audio_data["train"].select_columns(["audio", "text"])
audio_data["test"] = audio_data["test"].select_columns(["audio", "text"])

print(audio_data)

DatasetDict({
    train: IterableDataset({
        features: ['audio', 'text'],
        num_shards: 1
    })
    test: IterableDataset({
        features: ['audio', 'text'],
        num_shards: 2
    })
})


## 2. Feature Extractor, Tokenizer and Processor
* The ASR pipeline can be de-composed into three stages:
  1. The feature extractor which pre-processes the raw audio-inputs to log-mel spectrograms
  2. The model which performs the sequence-to-sequence mapping
  3. The tokenizer which post-processes the predicted tokens to text

* In Transformers, the Whisper model has an associated feature extractor and tokenizer, called `WhisperFeatureExtractor` and `WhisperTokenizer` respectively. These two objects are wrapped under a single class, called the `WhisperProcessor`.

* When performing multilingual fine-tuning, we need to set the "language" and "task" when instantiating the processor. The "language" should be set to the source audio language, and the task to "transcribe" for speech recognition or "translate" for speech translation. These arguments modify the behaviour of the tokenizer, and should be set correctly to ensure the target labels are encoded properly.

In [8]:
# Check all possible languages supported by Whisper
from transformers.models.whisper.tokenization_whisper import TO_LANGUAGE_CODE

TO_LANGUAGE_CODE

{'english': 'en',
 'chinese': 'zh',
 'german': 'de',
 'spanish': 'es',
 'russian': 'ru',
 'korean': 'ko',
 'french': 'fr',
 'japanese': 'ja',
 'portuguese': 'pt',
 'turkish': 'tr',
 'polish': 'pl',
 'catalan': 'ca',
 'dutch': 'nl',
 'arabic': 'ar',
 'swedish': 'sv',
 'italian': 'it',
 'indonesian': 'id',
 'hindi': 'hi',
 'finnish': 'fi',
 'vietnamese': 'vi',
 'hebrew': 'he',
 'ukrainian': 'uk',
 'greek': 'el',
 'malay': 'ms',
 'czech': 'cs',
 'romanian': 'ro',
 'danish': 'da',
 'hungarian': 'hu',
 'tamil': 'ta',
 'norwegian': 'no',
 'thai': 'th',
 'urdu': 'ur',
 'croatian': 'hr',
 'bulgarian': 'bg',
 'lithuanian': 'lt',
 'latin': 'la',
 'maori': 'mi',
 'malayalam': 'ml',
 'welsh': 'cy',
 'slovak': 'sk',
 'telugu': 'te',
 'persian': 'fa',
 'latvian': 'lv',
 'bengali': 'bn',
 'serbian': 'sr',
 'azerbaijani': 'az',
 'slovenian': 'sl',
 'kannada': 'kn',
 'estonian': 'et',
 'macedonian': 'mk',
 'breton': 'br',
 'basque': 'eu',
 'icelandic': 'is',
 'armenian': 'hy',
 'nepali': 'ne',
 'mongol

In [17]:
# Load the processor from the pre-trained checkpoint
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small", language="en", task="transcribe"
)

In [18]:
audio_data["train"].features

{'audio': Audio(sampling_rate=16000, decode=True, stream_index=None),
 'text': Value('string')}

In [19]:
# Downsample it to 16kHz prior to passing it to the Whisper
from datasets import Audio

sampling_rate = processor.feature_extractor.sampling_rate

audio_data["train"] = audio_data["train"].cast_column("audio", Audio(sampling_rate=sampling_rate))
audio_data["test"] = audio_data["test"].cast_column("audio", Audio(sampling_rate=sampling_rate))

print(audio_data)

DatasetDict({
    train: IterableDataset({
        features: ['audio', 'text'],
        num_shards: 1
    })
    test: IterableDataset({
        features: ['audio', 'text'],
        num_shards: 2
    })
})


In [20]:
# Write a function to prepare our data ready for the model

def prepare_dataset(example):
  audio = example["audio"]

  processed = processor(
      audio=audio["array"],
      sampling_rate=audio["sampling_rate"],
      text=example["text"],
  )

  example["input_features"] = processed["input_features"]
  example["labels"] = processed["labels"]

  # compute input length of audio sample in seconds
  example["input_length"] = len(audio["array"]) / audio["sampling_rate"]

  return example

In [21]:
# For IterableDataset: no num_proc, remove_columns as a list
audio_data = DatasetDict({
    split: ds.map(prepare_dataset, remove_columns=["audio", "text"])
    for split, ds in audio_data.items()
})

# Check the first sample
sample = next(iter(audio_data["train"]))
print(sample.keys())
print(sample["input_features"].shape)  # should be (1, 80, 3000) for Whisper
print(sample["labels"][:10])           # should be token ids
print(sample["input_length"])          # should be a float in seconds

dict_keys(['input_features', 'labels', 'input_length'])
(1, 80, 3000)
[50258, 50259, 50359, 50363, 32, 42930, 11944, 6783, 13513, 2195]
8.26


In [22]:
# Filter any training data with audio samples longer than 30s

max_input_length = 30.0

def is_audio_in_length_range(length):
    return length < max_input_length

audio_data["train"] = audio_data["train"].filter(
    is_audio_in_length_range,
    input_columns=["input_length"]
)

# Count after
# after = sum(1 for _ in audio_data["train"])
# print(f"After filtering: {after} samples")

## 3. Training and Evaluation

In [23]:
# Define a Data Collator
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
  processor: Any

  def __call__(
      self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
      ) -> Dict[str, torch.Tensor]:

      # split inputs and labels since they have to be of different lengths and need different padding methods
      # first treat the audio inputs by simply returning torch tensors
      input_features = [
          {"input_features": feature["input_features"][0]} for feature in features
      ]
      batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

      # get the tokenized label sequences
      label_features = [{"input_ids": feature["labels"]} for feature in features]

      # pad the labels to max length
      labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

      # replace padding with -100 to ignore loss correctly
      labels = labels_batch["input_ids"].masked_fill(
          labels_batch.attention_mask.ne(1), -100
      )

      # if bos token is appended in previous tokenization step,
      # cut bos token here as it's append later anyways
      if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
        labels = labels[:, 1:]

      batch["labels"] = labels

      return batch





In [27]:
#  Initialise the data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [29]:
!pip install --upgrade evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 66.9 MB/s eta 0:00:00


In [30]:
# Evaluation Metrics
import evaluate

metric = evaluate.load("wer")

In [33]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # compute orthographic wer
    wer_ortho = 100 * metric.compute(predictions=pred_str, references=label_str)

    # compute normalised WER
    pred_str_norm = [normalizer(pred) for pred in pred_str]
    label_str_norm = [normalizer(label) for label in label_str]
    # filtering step to only evaluate the samples that correspond to non-zero references:
    pred_str_norm = [
        pred_str_norm[i] for i in range(len(pred_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm = [
        label_str_norm[i]
        for i in range(len(label_str_norm))
        if len(label_str_norm[i]) > 0
    ]

    wer = 100 * metric.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer_ortho": wer_ortho, "wer": wer}

In [34]:
# Load a Pre-Trained Checkpoint

from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [35]:
from functools import partial

# disable cache during training since it's incompatible with gradient checkpointing
model.config.use_cache = False

# set language and task for generation and re-enable cache
model.generate = partial(
    model.generate, language="en", task="transcribe", use_cache=True
)

In [41]:
# Define the Training Configuration
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/whisper-small-gigaspeech",  # name on my colab
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=50,
    max_steps=500,  # increase to 4000 if you have your own GPU or a Colab paid plan
    gradient_checkpointing=True,
    fp16=True,
    fp16_full_eval=True,
    eval_strategy="steps",
    per_device_eval_batch_size=16,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

In [43]:
# Forward the trianing arguments to the Trainer
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=audio_data["train"],
    eval_dataset=audio_data["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

In [44]:
# Launch training

trainer.train()

Step,Training Loss,Validation Loss,Wer Ortho,Wer
500,0.000199,0.641215,63.793103,30.326296


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=500, training_loss=0.14379194252379238, metrics={'train_runtime': 1897.5754, 'train_samples_per_second': 4.216, 'train_steps_per_second': 0.263, 'total_flos': 2.06280844148736e+18, 'train_loss': 0.14379194252379238, 'epoch': 71.006})